In [1]:
"""
RAG with Diffusion Models
=========================
Primary Reference : Blattmann et al., "Retrieval-Augmented Diffusion Models" (RDM)
                   arXiv: 2204.11824  (NeurIPS 2022, CompVis / LMU Munich)
                   Code:  https://github.com/CompVis/retrieval-augmented-diffusion-models

Secondary References:
    Re-Imagen    : Chen et al. 2022   -- pixel-space with text-image pair retrieval
    KNN-Diffusion: Sheynin et al.2022 -- text-free kNN diffusion with CLIP
    Retrieve&Fuse: Blattmann et al.2022-- prevent CLIP info loss via latent concat
    ImageRAG     : arXiv 2502.09411   -- training-free retrieval for frozen models
    Cross-modal  : arXiv 2505.21956   -- sub-dimensional dual decomposition retrieval
    ImageRAGTurbo: arXiv 2602.12640   -- single-step few-step diffusion + RAG

Architecture: FIVE configurations covering the full RAG-diffusion design space:
    (1) Baseline Diffusion -- stable diffusion with text-only CLIP conditioning
    (2) CLIP kNN Retrieval + Cross-Attention Conditioning (core RDM)
    (3) Retrieve&Fuse -- latent-space concatenation of retrieved & noised images
    (4) Text-Guided Adaptive Retrieval (ImageRAG-style, training-free)
    (5) Full Multi-Modal Retrieval-Guided Diffusion [BEST] -- CLIP kNN +
        text-image pair guidance + adaptive retrieval weight interpolation

=============================================================================
FUNDAMENTAL DIFFERENCE FROM TEXT RAG: WHAT IS BEING RETRIEVED
=============================================================================
In text-based RAG (Pipelines 1-25 in this series):
    Query:  text string
    Index:  text chunks embedded by bi-encoder
    Output: text chunks injected into LLM context window

In RAG with Diffusion Models:
    Query:  CLIP embedding of text prompt (or input image)
    Index:  CLIP image embeddings of an external visual database
    Output: neighbor image embeddings injected into DENOISING U-NET
            via cross-attention at every denoising timestep T -> 0

This is NOT optional context prepended before generation. The retrieved
neighbor embeddings are conditioning signals that the U-Net reads at
EVERY step of the iterative denoising process (T=1000 steps typically).
The retrieval influences the ENTIRE generation trajectory, not just the
input prompt.

=============================================================================
THE SEMI-PARAMETRIC ARGUMENT: WHY NOT JUST SCALE THE MODEL?
=============================================================================
The dominant approach to improving image generation quality is to SCALE
the parametric model: more parameters = more visual knowledge compressed
into weights. This requires exponentially more FLOPS for training and
prohibitively larger GPU memory for inference.

RDM's argument (Blattmann et al., 2022): the model does NOT need to memorize
all visual content in its weights. It needs to learn COMPOSITION -- how to
combine and stylize visual elements. The specific visual content (textures,
object identity, fine-grained details) can be stored in an external database
and retrieved at generation time.

Semi-parametric factorization:
    p(x | c) = p_theta(x | c, NN_k(c, D))
    where:
        p_theta = small diffusion model (parametric, fixed after training)
        c       = text/class conditioning signal
        NN_k    = top-k nearest neighbors from external database D
        D       = large visual database (non-parametric, swappable at inference)

Key insight: swapping D at inference time (without changing model weights)
changes the MODEL's CAPABILITIES. The same trained model can generate:
    - Natural images (D = OpenImages)
    - Artistic paintings (D = ArtBench)
    - Domain-specific images (D = medical, satellite, microscopy)
This is impossible with purely parametric models without retraining.

=============================================================================
CLIP AS THE SHARED QUERY/DATABASE EMBEDDING SPACE
=============================================================================
CLIP (Radford et al., 2021) embeds both images and text into a shared
512-or-768-dimensional space where:
    cosine_sim(CLIP_text("a dog"), CLIP_image(dog_photo)) is high.

This shared space enables two critical capabilities for RDM:
    (1) TEXT-DRIVEN RETRIEVAL from an IMAGE database:
        Query: CLIP_text("impressionist painting of a sunset")
        Database: CLIP_image embeddings of ArtBench
        FAISS cosine search returns impressionist sunset paintings.
        No text labels needed in the database.

    (2) ZERO-SHOT DOMAIN ADAPTATION:
        The model was trained to condition on image CLIP embeddings.
        At inference with a text query:
            CLIP_text(prompt) lives in the same space as CLIP_image embeddings.
            So we can DIRECTLY condition the model on CLIP_text(prompt) as if
            it were a retrieved image embedding.
        This is why the same model handles both retrieval-based and
        text-only conditioning without modification.

=============================================================================
CONDITIONING MECHANISM: HOW NEIGHBORS ENTER THE U-NET
=============================================================================
The U-Net receives the k neighbor CLIP embeddings via a learned conditioning
pathway. In RDM, the k neighbor embeddings are concatenated into a single
"retrieval token sequence" of shape (k, embedding_dim) and injected via
cross-attention at every U-Net attention block. Specifically:

    For each U-Net attention block at resolution (H_l, W_l):
        spatial_features Q = Linear(features)     shape: (H_l*W_l, d_attn)
        neighbors K = Linear(neighbor_embeddings)  shape: (k, d_attn)
        neighbors V = Linear(neighbor_embeddings)  shape: (k, d_attn)
        attended = softmax(QK^T / sqrt(d_attn)) * V
        features = features + attended

This means at every spatial location at every resolution, the U-Net's
spatial features attend to ALL k neighbor embeddings simultaneously.
The model learns to extract different aspects from different neighbors:
one neighbor might provide color information, another texture, another
object identity. This attention mechanism is what allows the model to
synthesize a NEW image that is RELATED to but NOT IDENTICAL to the neighbors.

=============================================================================
RETRIEVE&FUSE: PREVENTING CLIP EMBEDDING INFORMATION LOSS
=============================================================================
A limitation of conditioning on CLIP embeddings is the information bottleneck:
CLIP compresses a 256x256x3 image into a 512-float vector. Fine-grained
texture details, exact colors, and precise spatial structures are lost.

Retrieve&Fuse (Blattmann et al., 2022 variant) addresses this by concatenating
the retrieved noised images DIRECTLY into the U-Net input, rather than using
CLIP embeddings as conditioning signals. At denoising timestep t:
    - Retrieved images are noised to the same level as the query: x^(i)_t = noise(x^(i), t)
    - Concatenated spatially: input_t = concat([x_t, x^(1)_t, ..., x^(k)_t], dim=channels)
    - U-Net processes the concatenated input with self-attention across all spatial positions
This allows FULL pixel-level information from the retrieved images to flow
into the denoising process, at the cost of k-fold larger U-Net input channels.

=============================================================================
IMPLEMENTATION NOTE
=============================================================================
This pipeline implements the INFERENCE-TIME behaviors of all five configurations
using Stable Diffusion v2 (Hugging Face diffusers) as the base model, CLIP ViT-L/14
for embedding, and FAISS for approximate nearest-neighbor search. The full
RDM training loop (fine-tuning the U-Net on retrieved-conditioned data) is not
implemented here -- that requires CompVis's training scripts and a large visual
database. All five configs here demonstrate the retrieval+conditioning behaviors
at inference time, which is the production-relevant use case (you have a trained
model, you swap in different retrieval databases).

"""

'\nRAG with Diffusion Models\n=========================\nPrimary Reference : Blattmann et al., "Retrieval-Augmented Diffusion Models" (RDM)\n                   arXiv: 2204.11824  (NeurIPS 2022, CompVis / LMU Munich)\n                   Code:  https://github.com/CompVis/retrieval-augmented-diffusion-models\n\nSecondary References:\n    Re-Imagen    : Chen et al. 2022   -- pixel-space with text-image pair retrieval\n    KNN-Diffusion: Sheynin et al.2022 -- text-free kNN diffusion with CLIP\n    Retrieve&Fuse: Blattmann et al.2022-- prevent CLIP info loss via latent concat\n    ImageRAG     : arXiv 2502.09411   -- training-free retrieval for frozen models\n    Cross-modal  : arXiv 2505.21956   -- sub-dimensional dual decomposition retrieval\n    ImageRAGTurbo: arXiv 2602.12640   -- single-step few-step diffusion + RAG\n\nArchitecture: FIVE configurations covering the full RAG-diffusion design space:\n    (1) Baseline Diffusion -- stable diffusion with text-only CLIP conditioning\n    (2) 

In [2]:
import os
import io
import sys
import time
import json
import logging
import hashlib
import urllib.request
import urllib.parse
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any, Union

import numpy as np
from PIL import Image

In [3]:
import torch
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionImg2ImgPipeline,
    DDIMScheduler,
)
from langchain_core.messages import HumanMessage

from huggingface_hub import hf_hub_download
import faiss


C:\Users\dhanu\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0526 07:35:44.225000 12972 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:


# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("rag_diffusion")


In [34]:

# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class RAGDiffusionConfig:
    """
    Centralized configuration for the RAG-Diffusion pipeline.

    SD_MODEL_ID ("stabilityai/stable-diffusion-2-1-base"):
        Stable Diffusion v2.1-base (512px) from Stability AI.
        ~5.2 GB VRAM at float16 on GPU, ~10 GB RAM on CPU.
        We use the -base variant (512x512) for development/demo.
        Production: upgrade to "stabilityai/stable-diffusion-2-1" (768px).

    CLIP_MODEL_ID ("openai/clip-vit-large-patch14"):
        CLIP ViT-L/14 from OpenAI. Embedding dim = 768.
        The SAME CLIP model used to build all retrieval databases.
        CRITICAL: database and query embeddings MUST be from the same
        CLIP model variant. Mixing ViT-L/14 and ViT-B/32 gives random results.

    N_NEIGHBORS (k=5):
        Number of nearest neighbors to retrieve from the visual database.
        RDM paper uses k in {1, 2, 4, 8, 16, 20} in ablations.
        At k=5: meaningful diversity of references while keeping cross-attention
        manageable. k=20 (max supported by RDM official weights) gives richer
        conditioning at the cost of higher memory in the cross-attention layers.

    RETRIEVAL_WEIGHT (0.6):
        In Config 5 (adaptive fusion), the interpolation weight between
        text-only generation (weight=0) and fully retrieved conditioning (weight=1).
        At 0.6: the retrieval contributes 60% of the conditioning signal,
        text prompt contributes 40%. This balance preserves prompt fidelity
        while incorporating visual references.
        Adjust per task: conceptual prompts (abstract art) -> 0.4 (more text)
                         object identity generation -> 0.8 (more retrieval)

    NUM_INFERENCE_STEPS (50):
        DDIM sampling steps. 50 is the standard production setting.
        For demo/development: use 20 steps (4x faster, slightly lower quality).
        RDM conditioning is stable across step counts because the retrieved
        embeddings are injected at EVERY step.

    GUIDANCE_SCALE (7.5):
        Classifier-free guidance (CFG) scale for text conditioning.
        At 7.5: strong text adherence balanced against image diversity.
        Lower (5.0): more creative, less literal interpretation of prompt.
        Higher (12.0): very literal, may reduce visual diversity.

    IMAGE_SIZE (512):
        Output image resolution in pixels (square).
        Must match SD_MODEL_ID's training resolution.
        SD v2.1-base: 512. SD v2.1 (full): 768.

    RETRIEVAL_DATABASE_DIR:
        Directory containing the pre-built visual retrieval database.
        Structure: retrieval_db/
                       embeddings.npy     -- (N, 768) CLIP image embeddings
                       paths.json         -- list of N image file paths / URLs
                       faiss_index.bin    -- FAISS inner-product (cosine) index
        Built once from a set of reference images using build_retrieval_database().

    DEMO_DATABASE_URLS:
        Small demo dataset: open-licensed images from Unsplash/WikiCommons.
        For production: replace with domain-specific high-quality image corpus.
        Database size scaling: FAISS HNSW index handles 100M images on a
        single machine. For larger: FAISS IVF (inverted file) + PQ compression.
    """

    # --- Diffusion Model --------------------------------------------------
    SD_MODEL_ID:         str   = "runwayml/stable-diffusion-v1-5"
    CLIP_MODEL_ID:       str   = "openai/clip-vit-large-patch14"
    DEVICE:              str   = "cuda" if torch.cuda.is_available() else "cpu"
    DTYPE:               torch.dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    # --- Retrieval Parameters --------------------------------------------
    N_NEIGHBORS:         int   = 5     # k in kNN retrieval
    N_NEIGHBORS_MAX:     int   = 10    # max for over-retrieval + reranking
    RETRIEVAL_WEIGHT:    float = 0.6   # Config 5 interpolation weight
    RETRIEVAL_DB_DIR:    str   = "./rag_diffusion_db"

    # --- Diffusion Sampling ----------------------------------------------
    NUM_INFERENCE_STEPS: int   = 50
    GUIDANCE_SCALE:      float = 7.5
    IMAGE_SIZE:          int   = 512
    IMG2IMG_STRENGTH:    float = 0.8   # Config 3 Retrieve&Fuse noise level

    # --- Output ----------------------------------------------------------
    OUTPUT_DIR: str = "./rag_diffusion_outputs"

    # --- Demo Database: open-licensed diverse reference images -----------
    # In production: replace with domain-specific images.
    # These are Wikipedia public domain images for demo.
    DEMO_IMAGE_URLS: List[Tuple[str, str]] = [
        # (filename, url)  -- public domain / CC0 / CC-BY images
        ("sunset_mountains.jpg", "https://picsum.photos/seed/sunset_mountains/768/512"),
        ("forest_path.jpg",      "https://picsum.photos/seed/forest_path/768/512"),
        ("ocean_waves.jpg",      "https://picsum.photos/seed/ocean_waves/768/512"),
        ("city_night.jpg",       "https://picsum.photos/seed/city_night/768/512"),
        ("abstract_art.jpg",     "https://picsum.photos/seed/abstract_art/768/512"),
        ("portrait_painting.jpg","https://picsum.photos/seed/portrait_painting/768/512"),
        ("landscape_oil.jpg",    "https://picsum.photos/seed/landscape_oil/768/512"),
        ("architecture.jpg",     "https://picsum.photos/seed/architecture/768/512"),
    ]

In [6]:

# ===========================================================================
# SECTION 2 -- DATA STRUCTURES
# ===========================================================================

@dataclass
class RetrievedNeighbor:
    """
    One retrieved nearest-neighbor image from the visual database.

    index:         position in the FAISS index (0..N-1)
    path:          file path or URL of the original image
    cosine_sim:    CLIP cosine similarity score to the query (0..1)
    clip_embedding: the CLIP image embedding (768-dim) of this neighbor
    image:         loaded PIL Image (lazy-loaded on demand)
    """
    index:          int
    path:           str
    cosine_sim:     float
    clip_embedding: Optional[np.ndarray] = None
    image:          Optional[Image.Image] = None


In [7]:

@dataclass
class RetrievalDatabase:
    """
    The visual retrieval database used by all RAG-Diffusion configs.

    embeddings:  np.array of shape (N, 768) -- CLIP embeddings of all database images.
                 L2-normalized for cosine similarity via inner product.
    paths:       list of N image file paths or URLs.
    faiss_index: FAISS IndexFlatIP (inner product = cosine similarity for L2-normed vecs).
    """
    embeddings:  np.ndarray
    paths:       List[str]
    faiss_index: Any   # faiss.IndexFlatIP

    def n_images(self) -> int:
        return len(self.paths)



In [8]:

@dataclass
class GenerationResult:
    """
    Result of one generation run for a given config.

    prompt:          text prompt used.
    strategy:        config name.
    generated_image: PIL Image of the generated output.
    neighbors:       list of retrieved neighbors used for conditioning.
    retrieval_ms:    time spent on CLIP embedding + FAISS search.
    generation_ms:   time spent on diffusion denoising.
    total_ms:        end-to-end latency.
    """
    prompt:          str
    strategy:        str
    generated_image: Optional[Image.Image] = None
    neighbors:       List[RetrievedNeighbor] = field(default_factory=list)
    retrieval_ms:    float = 0.0
    generation_ms:   float = 0.0
    total_ms:        float = 0.0
    metadata:        Dict[str, Any] = field(default_factory=dict)

    def save(self, output_dir: str, suffix: str = "") -> Path:
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        slug = self.prompt[:40].replace(" ", "_").replace("/", "-")
        fname = f"{self.strategy}_{slug}{suffix}.png"
        if self.generated_image:
            self.generated_image.save(out / fname)
            logger.info("Saved: %s", out / fname)
        return out / fname

    def print_summary(self) -> None:
        print(f"\n{'='*70}")
        print(f"STRATEGY: {self.strategy}")
        print(f"PROMPT  : {self.prompt[:68]}")
        if self.neighbors:
            print(f"Neighbors ({len(self.neighbors)}):")
            for n in self.neighbors[:3]:
                print(f"  [{n.index}] sim={n.cosine_sim:.3f}  {Path(n.path).name[:45]}")
        print(f"Latency : ret={self.retrieval_ms:.0f}ms  gen={self.generation_ms:.0f}ms  "
              f"total={self.total_ms:.0f}ms")
        print("=" * 70 + "\n")


In [25]:


# ===========================================================================
# SECTION 3 -- CLIP EMBEDDING INFRASTRUCTURE
# ===========================================================================

class CLIPEmbedder:
    """
    Wrapper around HuggingFace CLIP for embedding text and images.

    Handles:
        - Text embedding: CLIP_text(prompt) -> (768,) L2-normalized
        - Image embedding: CLIP_image(PIL_image) -> (768,) L2-normalized
        - Batch image embedding: for database construction

    The L2-normalization is critical: FAISS IndexFlatIP (inner product)
    on L2-normalized vectors computes exact cosine similarity.
    Without normalization, inner product is NOT cosine similarity.

    Embedding dimension: 768 for ViT-L/14, 512 for ViT-B/32.
    We use ViT-L/14 (768-dim) for richer visual semantics.
    """

    def __init__(self, model_id: str, device: str, dtype: torch.dtype) -> None:
        logger.info("Loading CLIP: %s on %s ...", model_id, device)
        self.device = device
        self.dtype  = dtype
        self.model  = CLIPModel.from_pretrained(model_id).to(device)
        self.proc   = CLIPProcessor.from_pretrained(model_id)
        self.model.eval()

    @staticmethod
    def _to_tensor(output: Any) -> torch.Tensor:
        """Normalize CLIP outputs across transformers versions."""
        if torch.is_tensor(output):
            return output
        if hasattr(output, "image_embeds") and output.image_embeds is not None:
            return output.image_embeds
        if hasattr(output, "text_embeds") and output.text_embeds is not None:
            return output.text_embeds
        if hasattr(output, "pooler_output") and output.pooler_output is not None:
            return output.pooler_output
        if isinstance(output, (tuple, list)) and len(output) > 0 and torch.is_tensor(output[0]):
            return output[0]
        raise TypeError(f"Unsupported CLIP output type: {type(output)}")
        # Get embedding dimension from vision projection output
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 224, 224, device=device)
            test_out = self.model.get_image_features(pixel_values=dummy)
            test_out = self._to_tensor(test_out)
            self.embed_dim = int(test_out.shape[-1])
        logger.info("CLIP loaded. Embedding dim: %d", self.embed_dim)

    @torch.no_grad()
    def embed_text(self, text: Union[str, List[str]]) -> np.ndarray:
        """
        Embed text prompt(s) into CLIP text embedding space.

        Args:
            text: single string or list of strings.

        Returns:
            np.array of shape (N, embed_dim) or (embed_dim,) for single string.
            L2-normalized.
        """
        if isinstance(text, str):
            text = [text]
        inputs  = self.proc(text=text, return_tensors="pt", padding=True,
                            truncation=True, max_length=77).to(self.device)
        embeds  = self.model.get_text_features(**inputs)
        embeds  = self._to_tensor(embeds)
        embeds  = F.normalize(embeds, p=2, dim=-1)
        arr     = embeds.cpu().float().numpy()
        return arr[0] if len(arr) == 1 else arr

    @torch.no_grad()
    def embed_image(self, image: Union[Image.Image, List[Image.Image]]) -> np.ndarray:
        """
        Embed PIL image(s) into CLIP image embedding space.

        Args:
            image: PIL Image or list of PIL Images.

        Returns:
            np.array of shape (N, embed_dim) or (embed_dim,) for single image.
            L2-normalized.
        """
        single = isinstance(image, Image.Image)
        images = [image] if single else image
        inputs = self.proc(images=images, return_tensors="pt").to(self.device)
        embeds = self.model.get_image_features(**inputs)
        embeds = self._to_tensor(embeds)
        embeds = F.normalize(embeds, p=2, dim=-1)
        arr    = embeds.cpu().float().numpy()
        return arr[0] if single else arr

    @torch.no_grad()
    def embed_images_batch(self, images: List[Image.Image],
                           batch_size: int = 32) -> np.ndarray:
        """
        Batch embed multiple images for database construction.

        Processes images in batches of batch_size to avoid OOM.
        Progress logged every 100 images.

        Args:
            images:     list of PIL Images.
            batch_size: images per GPU batch.

        Returns:
            np.array of shape (len(images), embed_dim), L2-normalized.
        """
        all_embeds = []
        for i in range(0, len(images), batch_size):
            batch = images[i: i + batch_size]
            inputs = self.proc(images=batch, return_tensors="pt").to(self.device)
            embeds = self.model.get_image_features(**inputs)
            embeds = self._to_tensor(embeds)
            embeds = F.normalize(embeds, p=2, dim=-1)
            all_embeds.append(embeds.cpu().float().numpy())
            if i % 100 == 0:
                logger.info("  Embedded %d / %d images ...", i, len(images))
        return np.concatenate(all_embeds, axis=0)


In [10]:

# ===========================================================================
# SECTION 4 -- RETRIEVAL DATABASE CONSTRUCTION AND SEARCH
# ===========================================================================

def download_demo_images(config: RAGDiffusionConfig) -> List[Tuple[str, Image.Image]]:
    """
    Download demo images from Wikipedia public domain URLs.

    Returns list of (path, PIL Image) tuples.
    Images are cached to config.RETRIEVAL_DB_DIR/images/.
    """
    img_dir = Path(config.RETRIEVAL_DB_DIR) / "images"
    img_dir.mkdir(parents=True, exist_ok=True)

    images: List[Tuple[str, Image.Image]] = []
    for fname, url in config.DEMO_IMAGE_URLS:
        dest = img_dir / fname
        if not dest.exists():
            try:
                logger.info("Downloading: %s", fname)
                req = urllib.request.Request(url,
                    headers={"User-Agent": "RAGDiffusion/1.0 (demo)"})
                with urllib.request.urlopen(req, timeout=30) as resp:
                    data = resp.read()
                dest.write_bytes(data)
                time.sleep(0.5)
            except Exception as exc:
                logger.warning("Failed to download %s: %s", fname, exc)
                continue

        try:
            img = Image.open(dest).convert("RGB").resize((256, 256), Image.LANCZOS)
            images.append((str(dest), img))
        except Exception as exc:
            logger.warning("Failed to load %s: %s", dest, exc)

    logger.info("Demo images loaded: %d", len(images))
    return images



In [11]:

def build_retrieval_database(
    image_data: List[Tuple[str, Image.Image]],
    clip: CLIPEmbedder,
    config: RAGDiffusionConfig,
) -> RetrievalDatabase:
    """
    Build the FAISS visual retrieval database from a set of reference images.

    Steps:
        1. Embed all images using CLIP (batch processing).
        2. L2-normalize embeddings (for cosine similarity via inner product).
        3. Build FAISS IndexFlatIP (exact inner product search).
           For production with >1M images: use IndexHNSWFlat for approximate
           search with sub-linear query time.
        4. Persist to disk as embeddings.npy + paths.json + faiss_index.bin.

    The database is the NON-PARAMETRIC MEMORY in the semi-parametric factorization.
    All visual knowledge lives here. Swapping this database changes what the
    model can generate WITHOUT changing the model weights.

    Args:
        image_data: list of (path, PIL Image) pairs.
        clip:       CLIPEmbedder for embedding.
        config:     RAGDiffusionConfig.

    Returns:
        Built RetrievalDatabase.
    """
    db_dir = Path(config.RETRIEVAL_DB_DIR)
    db_dir.mkdir(parents=True, exist_ok=True)

    emb_file   = db_dir / "embeddings.npy"
    paths_file = db_dir / "paths.json"
    idx_file   = db_dir / "faiss_index.bin"

    # Load from cache if all three files exist
    if emb_file.exists() and paths_file.exists() and idx_file.exists():
        logger.info("Loading retrieval database from cache: %s", db_dir)
        embeddings = np.load(str(emb_file))
        with open(paths_file) as f:
            paths = json.load(f)
        faiss_index = faiss.read_index(str(idx_file))
        logger.info("Database loaded: %d images, embedding_dim=%d",
                    len(paths), embeddings.shape[1])
        return RetrievalDatabase(embeddings=embeddings, paths=paths,
                                 faiss_index=faiss_index)

    logger.info("Building retrieval database from %d images ...", len(image_data))

    paths  = [p for p, _ in image_data]
    images = [img for _, img in image_data]

    # Embed all images in batches
    t0 = time.perf_counter()
    embeddings = clip.embed_images_batch(images, batch_size=16)
    embed_ms   = (time.perf_counter() - t0) * 1000
    logger.info("  Embeddings computed: shape=%s in %.0f ms",
                embeddings.shape, embed_ms)

    # Ensure L2 normalization for cosine similarity
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings = embeddings / (norms + 1e-8)

    # Build FAISS inner product index (cosine sim for normalized vectors)
    dim = embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dim)
    faiss_index.add(embeddings.astype(np.float32))
    logger.info("  FAISS index built: %d vectors, dim=%d", faiss_index.ntotal, dim)

    # Persist
    np.save(str(emb_file), embeddings)
    with open(paths_file, "w") as f:
        json.dump(paths, f)
    faiss.write_index(faiss_index, str(idx_file))
    logger.info("Database saved to %s", db_dir)

    return RetrievalDatabase(embeddings=embeddings, paths=paths,
                             faiss_index=faiss_index)


In [12]:

def retrieve_neighbors(
    query_embedding: np.ndarray,
    db: RetrievalDatabase,
    k: int,
    load_images: bool = False,
) -> List[RetrievedNeighbor]:
    """
    Retrieve k nearest neighbors from the visual database.

    Uses FAISS IndexFlatIP for exact inner product (= cosine similarity
    for L2-normalized vectors). For L2-normalized vectors:
        inner_product(q, d) = ||q||_2 * ||d||_2 * cos(angle) = cos(angle)
    because both q and d are unit vectors.

    Args:
        query_embedding: (embed_dim,) query CLIP embedding, L2-normalized.
        db:              RetrievalDatabase.
        k:               number of neighbors to retrieve.
        load_images:     if True, load the actual PIL images of the neighbors.

    Returns:
        List of RetrievedNeighbor, sorted by descending cosine similarity.
    """
    q = query_embedding.reshape(1, -1).astype(np.float32)

    # FAISS returns (similarities, indices) with shape (1, k)
    sims, idxs = db.faiss_index.search(q, k)

    neighbors: List[RetrievedNeighbor] = []
    for rank, (idx, sim) in enumerate(zip(idxs[0], sims[0])):
        if idx < 0:  # FAISS returns -1 for empty results
            continue
        path = db.paths[idx]
        clip_emb = db.embeddings[idx].copy()

        pil_img = None
        if load_images:
            try:
                pil_img = Image.open(path).convert("RGB").resize((512, 512), Image.LANCZOS)
            except Exception:
                pass

        neighbors.append(RetrievedNeighbor(
            index=int(idx), path=path, cosine_sim=float(sim),
            clip_embedding=clip_emb, image=pil_img,
        ))

    return neighbors


In [13]:

# ===========================================================================
# SECTION 5 -- DIFFUSION PIPELINE BUILDERS
# ===========================================================================

def load_sd_pipeline(config: RAGDiffusionConfig) -> StableDiffusionPipeline:
    """
    Load Stable Diffusion v2.1-base pipeline.

    Uses float16 on CUDA for 2x memory reduction vs float32.
    DDIM scheduler for faster, deterministic sampling.
    """
    logger.info("Loading Stable Diffusion: %s ...", config.SD_MODEL_ID)
    pipe = StableDiffusionPipeline.from_pretrained(
        config.SD_MODEL_ID,
        torch_dtype=config.DTYPE,
        safety_checker=None,      # disabled for research use
        requires_safety_checker=False,
    )
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to(config.DEVICE)

    # Enable memory efficient attention if xformers available
    try:
        pipe.enable_xformers_memory_efficient_attention()
        logger.info("  xformers memory efficient attention enabled.")
    except Exception:
        logger.info("  xformers not available; using default attention.")

    # Enable attention slicing on CPU to reduce RAM usage
    if config.DEVICE == "cpu":
        pipe.enable_attention_slicing()

    logger.info("Stable Diffusion loaded on %s (dtype=%s)", config.DEVICE, config.DTYPE)
    return pipe



In [14]:

def load_img2img_pipeline(config: RAGDiffusionConfig) -> StableDiffusionImg2ImgPipeline:
    """
    Load Stable Diffusion img2img pipeline for Config 3 (Retrieve&Fuse).

    img2img takes an init_image (the retrieved neighbor) + a text prompt
    and denoises from a noised version of the init_image. This approximates
    the Retrieve&Fuse mechanism without requiring U-Net architecture changes.

    The strength parameter controls how much noise to add to the init_image
    before denoising: strength=0.0 returns the original image unchanged;
    strength=1.0 is equivalent to text-to-image generation.
    """
    logger.info("Loading SD img2img pipeline ...")
    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
        config.SD_MODEL_ID,
        torch_dtype=config.DTYPE,
        safety_checker=None,
        requires_safety_checker=False,
    )
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to(config.DEVICE)
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass
    if config.DEVICE == "cpu":
        pipe.enable_attention_slicing()
    return pipe



In [15]:


# ===========================================================================
# SECTION 6 -- FIVE RAG-DIFFUSION CONFIGURATIONS
# ===========================================================================

# ---------------------------------------------------------------------------
# CONFIG 1: Baseline -- Text-Only Stable Diffusion (no retrieval)
# ---------------------------------------------------------------------------

def run_config1_baseline_text_only(
    prompt: str,
    pipe: StableDiffusionPipeline,
    config: RAGDiffusionConfig,
    seed: int = 42,
) -> GenerationResult:
    """
    Configuration 1: Standard text-to-image generation (no retrieval).

    Pure parametric generation: all visual knowledge is encoded in the
    model weights. The text prompt is embedded by CLIP text encoder and
    used to condition the U-Net denoising process via cross-attention.

    This is the control condition. Its failures motivate the retrieval configs:
        - Novel fine-grained concepts (rare species, specific art styles,
          domain-specific objects) are poorly represented or hallucinated.
        - Generating out-of-distribution (OOD) content requires retraining.
        - No mechanism to incorporate user's specific reference images.

    LLM-parallel: Config 1 is the RAG-Diffusion equivalent of an LLM
    answering from parametric knowledge without any retrieved context.

    Compute: ~10-12s on A100 (50 DDIM steps, 512x512).
    """
    result = GenerationResult(prompt=prompt, strategy="Config1_Baseline_TextOnly")
    t0 = time.perf_counter()

    generator = torch.Generator(device=config.DEVICE).manual_seed(seed)

    logger.info("Config1: Generating '%s ...' (no retrieval)", prompt[:50])
    t_gen = time.perf_counter()
    out = pipe(
        prompt=prompt,
        num_inference_steps=config.NUM_INFERENCE_STEPS,
        guidance_scale=config.GUIDANCE_SCALE,
        height=config.IMAGE_SIZE,
        width=config.IMAGE_SIZE,
        generator=generator,
    )
    result.generation_ms = (time.perf_counter() - t_gen) * 1000
    result.generated_image = out.images[0]
    result.total_ms = (time.perf_counter() - t0) * 1000
    return result


In [16]:

# ---------------------------------------------------------------------------
# CONFIG 2: CLIP kNN Retrieval + Cross-Attention Conditioning (core RDM)
# ---------------------------------------------------------------------------

def run_config2_clip_knn_conditioning(
    prompt: str,
    pipe: StableDiffusionPipeline,
    clip: CLIPEmbedder,
    db: RetrievalDatabase,
    config: RAGDiffusionConfig,
    seed: int = 42,
) -> GenerationResult:
    """
    Configuration 2: CLIP kNN Retrieval with neighbor-augmented conditioning.

    This implements the CORE mechanism of Retrieval-Augmented Diffusion Models
    (Blattmann et al., NeurIPS 2022): query the external database with the
    CLIP text embedding of the prompt, retrieve k nearest visual neighbors,
    and use their CLIP embeddings to enrich the generation conditioning.

    APPROXIMATION NOTE:
    The full RDM architecture fine-tunes the U-Net's cross-attention layers
    to accept a combined (text + neighbor embeddings) conditioning sequence.
    Since we use an off-the-shelf SD v2.1 pipeline (NOT fine-tuned for
    neighbor conditioning), we implement this as a PROMPT AUGMENTATION:

        Original prompt:       "a sunset over mountains"
        Retrieved neighbors:   high cosine similarity to CLIP_text("a sunset over mountains")
                               e.g., "sunset_mountains.jpg" (sim=0.89)
                                     "landscape_oil.jpg"     (sim=0.82)
        Augmented prompt:      "a sunset over mountains, inspired by [visual style of
                                nearest reference images: warm colors, dramatic sky,
                                mountain silhouette]"
        + Negative prompt:     built from low-similarity neighbors (contrast signal)

    This approximation demonstrates the retrieval behavior and output diversity
    enabled by the database at the cost of not having the exact cross-attention
    conditioning from the trained RDM.

    For the actual RDM cross-attention conditioning: use the CompVis RDM weights
    (https://github.com/CompVis/retrieval-augmented-diffusion-models) with the
    knn2img.py inference script, which loads the trained U-Net conditioned
    on k neighbor CLIP embeddings.

    Key observable benefits vs Config 1:
        - Swapping the database changes the style/content domain without retraining.
        - The neighbors are logged in the trace for interpretability.
        - Retrieval adds ~50-200ms overhead (CLIP query + FAISS search), minimal.

    Args:
        prompt : Text prompt for generation.
        pipe   : Stable Diffusion pipeline.
        clip   : CLIPEmbedder for query embedding.
        db     : RetrievalDatabase (the external visual memory).
        config : RAGDiffusionConfig.
        seed   : Random seed for reproducibility.

    Returns:
        GenerationResult with retrieved neighbors attached.
    """
    result = GenerationResult(prompt=prompt, strategy="Config2_CLIP_kNN_Retrieval")
    t0 = time.perf_counter()

    # --- Step 1: Embed text query using CLIP text encoder -----------------
    t_ret = time.perf_counter()
    query_emb = clip.embed_text(prompt)  # shape: (768,), L2-normalized

    # --- Step 2: Retrieve k nearest visual neighbors from database ---------
    neighbors = retrieve_neighbors(
        query_embedding=query_emb, db=db,
        k=config.N_NEIGHBORS, load_images=False,
    )
    result.retrieval_ms = (time.perf_counter() - t_ret) * 1000
    result.neighbors    = neighbors

    logger.info(
        "Config2: Retrieved %d neighbors for '%s ...' "
        "(top sim=%.3f, ret=%.0fms)",
        len(neighbors), prompt[:40],
        neighbors[0].cosine_sim if neighbors else 0.0,
        result.retrieval_ms,
    )

    # --- Step 3: Augment prompt with retrieved visual context description --
    # This is the inference-time approximation of cross-attention neighbor conditioning.
    # The retrieved neighbor file names carry semantic signal (e.g., "sunset_mountains.jpg").
    neighbor_descriptions = []
    for n in neighbors[:3]:
        n_name  = Path(n.path).stem.replace("_", " ")
        sim_pct = int(n.cosine_sim * 100)
        neighbor_descriptions.append(f"{n_name} ({sim_pct}% match)")

    augmented_prompt = prompt
    if neighbor_descriptions:
        style_hint = ", ".join(neighbor_descriptions[:2])
        augmented_prompt = f"{prompt}, referencing visual concepts of: {style_hint}"

    # --- Step 4: Generate with retrieval-augmented prompt -----------------
    generator = torch.Generator(device=config.DEVICE).manual_seed(seed)
    t_gen = time.perf_counter()
    out = pipe(
        prompt=augmented_prompt,
        negative_prompt="blurry, low quality, distorted, unrealistic",
        num_inference_steps=config.NUM_INFERENCE_STEPS,
        guidance_scale=config.GUIDANCE_SCALE,
        height=config.IMAGE_SIZE,
        width=config.IMAGE_SIZE,
        generator=generator,
    )
    result.generation_ms  = (time.perf_counter() - t_gen) * 1000
    result.generated_image = out.images[0]
    result.metadata["augmented_prompt"] = augmented_prompt
    result.total_ms = (time.perf_counter() - t0) * 1000
    return result


In [17]:

# ---------------------------------------------------------------------------
# CONFIG 3: Retrieve&Fuse -- Latent Image Conditioning
# ---------------------------------------------------------------------------

def run_config3_retrieve_fuse(
    prompt: str,
    pipe_img2img: StableDiffusionImg2ImgPipeline,
    clip: CLIPEmbedder,
    db: RetrievalDatabase,
    config: RAGDiffusionConfig,
    seed: int = 42,
) -> GenerationResult:
    """
    Configuration 3: Retrieve&Fuse -- retrieved image as latent conditioning.

    This config implements the Retrieve&Fuse approach (Blattmann et al., 2022
    variant) where the retrieved image is used as a structural/content prior
    for generation rather than just as an embedding conditioning signal.

    Mechanism (implemented via img2img):
        1. Retrieve the SINGLE most similar image from the database (k=1).
        2. Use it as the init_image for img2img denoising.
        3. Add noise at level = strength (0.8 = 80% noise, 20% original).
        4. Denoise from the noised retrieved image guided by the text prompt.

    This is an approximation of the full Retrieve&Fuse architecture, which
    concatenates k noised retrieved images directly to the U-Net input:
        input_t = concat([query_noised_t, retrieved_noised_1_t, ..., k_t], dim_channels)
    The exact architecture requires a modified U-Net with 4*(1+k) input channels
    instead of the standard 4 input channels (for latent space img2img).

    Why this config is distinct from Config 2:
        Config 2 conditions on CLIP EMBEDDINGS (512/768-float vector):
            -- compressed representation, loses fine-grained texture/color
            -- conditioning happens via cross-attention (global influence)
            -- no structural copying from the retrieved image
        Config 3 conditions on the RETRIEVED IMAGE PIXELS (via noised latent):
            -- full pixel information preserved before noise injection
            -- conditioning happens via latent space initialization (spatial structure)
            -- the generated image inherits the spatial composition of the retrieved image
            -- use case: style transfer, structure-preserving generation

    Strength tuning:
        strength=0.3: 30% noise, output stays close to retrieved image (subtle refinement)
        strength=0.5: 50% noise, balanced style adoption + prompt alignment
        strength=0.8: 80% noise, mostly text-driven with retrieved image as weak prior
        strength=1.0: equivalent to text-to-image (no retrieval influence)

    Compute: ~12-15s (FAISS retrieval + img2img denoising + image load).
    """
    result = GenerationResult(prompt=prompt, strategy="Config3_Retrieve_Fuse_img2img")
    t0     = time.perf_counter()

    # --- Step 1: Retrieve single best neighbor (k=1) ----------------------
    t_ret = time.perf_counter()
    query_emb = clip.embed_text(prompt)
    neighbors = retrieve_neighbors(
        query_embedding=query_emb, db=db,
        k=1, load_images=True,   # load_images=True to get PIL image
    )
    result.retrieval_ms = (time.perf_counter() - t_ret) * 1000
    result.neighbors    = neighbors

    if not neighbors or neighbors[0].image is None:
        logger.warning("Config3: No neighbor image found. Falling back to text-only.")
        result.generated_image = Image.new("RGB", (config.IMAGE_SIZE, config.IMAGE_SIZE))
        result.total_ms = (time.perf_counter() - t0) * 1000
        return result

    init_image = neighbors[0].image.resize(
        (config.IMAGE_SIZE, config.IMAGE_SIZE), Image.LANCZOS
    )
    logger.info(
        "Config3: Retrieved init image '%s' (sim=%.3f, ret=%.0fms)",
        Path(neighbors[0].path).name, neighbors[0].cosine_sim,
        result.retrieval_ms,
    )

    # --- Step 2: img2img denoising from retrieved image -------------------
    generator = torch.Generator(device=config.DEVICE).manual_seed(seed)
    t_gen = time.perf_counter()
    out = pipe_img2img(
        prompt=prompt,
        image=init_image,
        strength=config.IMG2IMG_STRENGTH,
        num_inference_steps=config.NUM_INFERENCE_STEPS,
        guidance_scale=config.GUIDANCE_SCALE,
        generator=generator,
    )
    result.generation_ms   = (time.perf_counter() - t_gen) * 1000
    result.generated_image = out.images[0]
    result.metadata["init_image_path"] = neighbors[0].path
    result.metadata["strength"]        = config.IMG2IMG_STRENGTH
    result.total_ms = (time.perf_counter() - t0) * 1000
    return result



In [18]:

# ---------------------------------------------------------------------------
# CONFIG 4: Text-Guided Adaptive Retrieval (ImageRAG-style, training-free)
# ---------------------------------------------------------------------------

def run_config4_imagrag_adaptive(
    prompt: str,
    pipe: StableDiffusionPipeline,
    pipe_img2img: StableDiffusionImg2ImgPipeline,
    clip: CLIPEmbedder,
    db: RetrievalDatabase,
    config: RAGDiffusionConfig,
    seed: int = 42,
) -> GenerationResult:
    """
    Configuration 4: Text-Guided Adaptive Retrieval (ImageRAG-style).

    Inspired by ImageRAG (arXiv 2502.09411): a training-free retrieval-guided
    generation method that leverages the capabilities of EXISTING image
    conditioning models (e.g., SD IP-Adapter, ControlNet) without requiring
    RAG-specific fine-tuning.

    This config implements the core insight: instead of conditioning the
    diffusion model on CLIP embeddings of neighbors (which requires fine-tuning),
    we use an ADAPTIVE RETRIEVAL STRATEGY:

    Step 1: OVER-RETRIEVE (k=N_NEIGHBORS_MAX=10).
            Retrieve more candidates than needed.

    Step 2: RERANK by compound score:
            compound_score(n_i) = alpha * cosine_sim_to_prompt(n_i)
                                + (1-alpha) * diversity_score(n_i, selected_neighbors)
            where diversity_score penalizes neighbors too similar to already-selected ones.
            This produces a diverse-but-relevant top-k, avoiding redundant conditioning.

    Step 3: MULTI-STAGE GENERATION:
            Stage 1: Generate a DRAFT image using text-only (Config 1 style).
            Stage 2: Rerank retrieved neighbors by similarity to the DRAFT IMAGE
                     (not just the text prompt). This grounds the retrieval in
                     what the model actually generated, not just the abstract prompt.
            Stage 3: Use the top-1 reranked neighbor for img2img refinement (Config 3).

    This two-stage process mirrors the ImageRAG finding that retrieved images
    should complement the model's generation trajectory, not just the query text.

    Compute: ~15-20s (over-retrieval + draft generation + reranking + refinement).
    """
    result = GenerationResult(prompt=prompt,
                              strategy="Config4_ImageRAG_Adaptive_Retrieval")
    t0 = time.perf_counter()

    # --- Step 1: Over-retrieve top-N_NEIGHBORS_MAX candidates -------------
    t_ret = time.perf_counter()
    query_emb = clip.embed_text(prompt)
    candidates = retrieve_neighbors(
        query_embedding=query_emb, db=db,
        k=config.N_NEIGHBORS_MAX, load_images=True,
    )
    ret_ms = (time.perf_counter() - t_ret) * 1000

    # --- Step 2: Diverse-aware reranking of candidates --------------------
    def diversity_score(candidate_emb: np.ndarray,
                        selected_embs: List[np.ndarray]) -> float:
        """Minimum cosine similarity to already-selected neighbors. Higher = more diverse."""
        if not selected_embs:
            return 1.0
        sims = [float(np.dot(candidate_emb, sel)) for sel in selected_embs]
        return 1.0 - max(sims)   # diversity = 1 - max_similarity_to_selected

    alpha      = 0.7   # weight for text similarity vs diversity
    selected   = []
    selected_embs: List[np.ndarray] = []
    remaining  = list(candidates)

    while len(selected) < config.N_NEIGHBORS and remaining:
        scores = []
        for cand in remaining:
            if cand.clip_embedding is None:
                continue
            text_sim = cand.cosine_sim
            div_s    = diversity_score(cand.clip_embedding, selected_embs)
            compound = alpha * text_sim + (1 - alpha) * div_s
            scores.append((compound, cand))
        if not scores:
            break
        scores.sort(key=lambda x: x[0], reverse=True)
        best_cand = scores[0][1]
        selected.append(best_cand)
        if best_cand.clip_embedding is not None:
            selected_embs.append(best_cand.clip_embedding)
        remaining.remove(best_cand)

    result.retrieval_ms = ret_ms
    result.neighbors    = selected
    logger.info(
        "Config4: Diverse reranked %d candidates -> top %d "
        "(sims=[%s], ret=%.0fms)",
        len(candidates), len(selected),
        ", ".join(f"{s.cosine_sim:.3f}" for s in selected[:3]),
        ret_ms,
    )

    # --- Step 3a: Generate draft image (text-only) -----------------------
    generator = torch.Generator(device=config.DEVICE).manual_seed(seed)
    t_draft = time.perf_counter()
    draft_out = pipe(
        prompt=prompt,
        num_inference_steps=max(20, config.NUM_INFERENCE_STEPS // 2),  # fast draft
        guidance_scale=config.GUIDANCE_SCALE,
        height=config.IMAGE_SIZE, width=config.IMAGE_SIZE,
        generator=generator,
    )
    draft_image = draft_out.images[0]
    draft_ms    = (time.perf_counter() - t_draft) * 1000
    logger.info("Config4: Draft generated in %.0fms", draft_ms)

    # --- Step 3b: Rerank neighbors by similarity to DRAFT image ----------
    draft_emb = clip.embed_image(draft_image.resize((224, 224), Image.LANCZOS))
    for n in selected:
        if n.clip_embedding is not None:
            # Re-score: similarity to the ACTUAL draft image, not text prompt
            n.cosine_sim = float(np.dot(draft_emb, n.clip_embedding))

    selected.sort(key=lambda n: n.cosine_sim, reverse=True)
    best_neighbor = selected[0] if selected else None
    logger.info(
        "Config4: Post-draft reranked. Best neighbor: '%s' (sim=%.3f)",
        Path(best_neighbor.path).name if best_neighbor else "None",
        best_neighbor.cosine_sim if best_neighbor else 0.0,
    )

    # --- Step 3c: Refine draft with best neighbor via img2img -------------
    if best_neighbor and best_neighbor.image is not None:
        init_img = best_neighbor.image.resize(
            (config.IMAGE_SIZE, config.IMAGE_SIZE), Image.LANCZOS
        )
        # Blend draft and neighbor 50/50 as init image for refinement
        blended_init = Image.blend(draft_image, init_img, alpha=0.4)

        t_gen = time.perf_counter()
        out = pipe_img2img(
            prompt=prompt,
            image=blended_init,
            strength=0.5,   # moderate noise for refinement
            num_inference_steps=config.NUM_INFERENCE_STEPS,
            guidance_scale=config.GUIDANCE_SCALE,
            generator=torch.Generator(device=config.DEVICE).manual_seed(seed + 1),
        )
        result.generation_ms   = draft_ms + (time.perf_counter() - t_gen) * 1000
        result.generated_image = out.images[0]
    else:
        result.generation_ms   = draft_ms
        result.generated_image = draft_image

    result.metadata["draft_used"] = True
    result.total_ms = (time.perf_counter() - t0) * 1000
    return result



In [19]:


# ---------------------------------------------------------------------------
# CONFIG 5: Full Multi-Modal Retrieval-Guided Diffusion [BEST]
# ---------------------------------------------------------------------------

def run_config5_full_retrieval_guided(
    prompt: str,
    pipe: StableDiffusionPipeline,
    pipe_img2img: StableDiffusionImg2ImgPipeline,
    clip: CLIPEmbedder,
    db: RetrievalDatabase,
    config: RAGDiffusionConfig,
    seed: int = 42,
) -> GenerationResult:
    """
    Configuration 5: Full Multi-Modal Retrieval-Guided Diffusion [BEST].

    Combines the best elements from Configs 2-4:
        - Diverse-aware kNN retrieval (Config 4's over-retrieval + reranking)
        - CLIP embedding interpolation for prompt enrichment (Config 2)
        - Latent image conditioning for the best neighbor (Config 3)
        - Adaptive retrieval weight based on query-database alignment quality

    ADAPTIVE RETRIEVAL WEIGHT:
    The weight w controls how much the retrieved images influence generation:
        w = RETRIEVAL_WEIGHT * alignment_quality
        where alignment_quality = avg(top-k cosine similarities)

    If the database is a poor match for the prompt (all sims < 0.3):
        w is reduced toward 0 -- fallback to text-only generation.
    If the database is a strong match (sims > 0.7):
        w is increased toward RETRIEVAL_WEIGHT -- strong retrieval conditioning.
    This adaptive weighting prevents the retrieved neighbors from degrading
    generation when the database lacks relevant visual content.

    CROSS-MODAL EMBEDDING INTERPOLATION:
    Rather than using the text embedding alone OR a retrieved image embedding,
    we interpolate in CLIP space:
        combined_emb = (1 - w) * text_emb + w * avg(neighbor_embs)
    This combined embedding lives in the shared CLIP text/image space and
    represents a query that is simultaneously text-grounded and visually-grounded.
    We then retrieve a SECOND PASS of neighbors using this combined embedding,
    which finds images that are relevant to BOTH the text AND the initial
    visual neighbors -- effectively performing two-stage retrieval.

    GENERATION STRATEGY:
        1. Retrieve k neighbors with text query (first pass).
        2. Compute adaptive weight w from alignment quality.
        3. Interpolate: combined_emb = (1-w)*text_emb + w*avg(neighbor_embs)
        4. Retrieve k neighbors with combined_emb (second pass -- finer-grained).
        5. Use best second-pass neighbor for img2img initialization.
        6. Use augmented prompt (from first-pass neighbor descriptions) for text.
        7. Generate with strength=w (higher alignment -> more retrieval influence).

    This two-pass retrieval with adaptive weight mirrors the RealRAG and
    Cross-modal RAG approaches (arXiv 2505.21956) that perform iterative
    query refinement for retrieval-augmented diffusion.

    Compute: ~20-25s (two FAISS passes + draft + refinement).
    """
    result = GenerationResult(
        prompt=prompt,
        strategy="Config5_Full_MultiModal_RetrievalGuided [BEST]",
    )
    t0 = time.perf_counter()

    # --- FIRST PASS: Text-query retrieval --------------------------------
    t_ret1 = time.perf_counter()
    text_emb   = clip.embed_text(prompt)
    neighbors1 = retrieve_neighbors(
        query_embedding=text_emb, db=db,
        k=config.N_NEIGHBORS_MAX, load_images=False,
    )
    ret1_ms = (time.perf_counter() - t_ret1) * 1000

    # Compute alignment quality: average top-k cosine similarities
    top_sims = [n.cosine_sim for n in neighbors1[:config.N_NEIGHBORS]]
    alignment_quality = float(np.mean(top_sims)) if top_sims else 0.0
    adaptive_weight   = config.RETRIEVAL_WEIGHT * min(1.0, alignment_quality / 0.5)
    # Clamp: retrieval weight in [0.1, RETRIEVAL_WEIGHT]
    adaptive_weight   = max(0.1, min(config.RETRIEVAL_WEIGHT, adaptive_weight))

    logger.info(
        "Config5: First pass - alignment_quality=%.3f -> adaptive_weight=%.3f",
        alignment_quality, adaptive_weight,
    )

    # --- SECOND PASS: Combined embedding retrieval -----------------------
    if neighbors1 and any(n.clip_embedding is not None for n in neighbors1[:3]):
        valid_embs = [n.clip_embedding for n in neighbors1[:3]
                      if n.clip_embedding is not None]
        avg_neighbor_emb = np.mean(valid_embs, axis=0)
        avg_neighbor_emb = avg_neighbor_emb / (np.linalg.norm(avg_neighbor_emb) + 1e-8)

        # Interpolate in CLIP space
        combined_emb = (1.0 - adaptive_weight) * text_emb + adaptive_weight * avg_neighbor_emb
        combined_emb = combined_emb / (np.linalg.norm(combined_emb) + 1e-8)

        t_ret2 = time.perf_counter()
        neighbors2 = retrieve_neighbors(
            query_embedding=combined_emb, db=db,
            k=config.N_NEIGHBORS, load_images=True,
        )
        ret2_ms = (time.perf_counter() - t_ret2) * 1000
        logger.info(
            "Config5: Second pass (combined emb) - top sim=%.3f (%.0fms)",
            neighbors2[0].cosine_sim if neighbors2 else 0.0, ret2_ms,
        )
    else:
        neighbors2 = retrieve_neighbors(
            query_embedding=text_emb, db=db,
            k=config.N_NEIGHBORS, load_images=True,
        )
        ret2_ms = 0.0

    result.retrieval_ms = ret1_ms + ret2_ms
    result.neighbors    = neighbors2

    # --- Build augmented prompt from second-pass neighbors ----------------
    neighbor_hints = []
    for n in neighbors2[:2]:
        n_name = Path(n.path).stem.replace("_", " ")
        neighbor_hints.append(n_name)
    augmented_prompt = (
        f"{prompt}, with visual references: {', '.join(neighbor_hints)}"
        if neighbor_hints else prompt
    )

    # --- Generate: adapt strategy to alignment quality -------------------
    generator = torch.Generator(device=config.DEVICE).manual_seed(seed)

    if adaptive_weight < 0.2 or not neighbors2 or neighbors2[0].image is None:
        # Low alignment: text-only generation
        logger.info("Config5: Low alignment (w=%.2f) -> text-only generation", adaptive_weight)
        t_gen = time.perf_counter()
        out = pipe(
            prompt=augmented_prompt,
            num_inference_steps=config.NUM_INFERENCE_STEPS,
            guidance_scale=config.GUIDANCE_SCALE,
            height=config.IMAGE_SIZE, width=config.IMAGE_SIZE,
            generator=generator,
        )
        result.generated_image = out.images[0]
        result.generation_ms   = (time.perf_counter() - t_gen) * 1000
        result.metadata["generation_path"] = "text_only_fallback"
    else:
        # Good alignment: img2img with adaptive strength
        best_neighbor = neighbors2[0]
        init_img = best_neighbor.image.resize(
            (config.IMAGE_SIZE, config.IMAGE_SIZE), Image.LANCZOS
        )
        # Adaptive strength: higher retrieval weight -> lower noise (more neighbor influence)
        adaptive_strength = 1.0 - adaptive_weight * 0.5  # range: [0.5, 0.95]
        logger.info(
            "Config5: img2img with neighbor '%s', strength=%.2f, w=%.2f",
            Path(best_neighbor.path).name, adaptive_strength, adaptive_weight,
        )
        t_gen = time.perf_counter()
        out = pipe_img2img(
            prompt=augmented_prompt,
            image=init_img,
            strength=adaptive_strength,
            num_inference_steps=config.NUM_INFERENCE_STEPS,
            guidance_scale=config.GUIDANCE_SCALE,
            generator=generator,
        )
        result.generated_image = out.images[0]
        result.generation_ms   = (time.perf_counter() - t_gen) * 1000
        result.metadata.update({
            "generation_path":  "adaptive_retrieval_guided",
            "adaptive_weight":  adaptive_weight,
            "adaptive_strength":adaptive_strength,
            "augmented_prompt": augmented_prompt,
        })

    result.total_ms = (time.perf_counter() - t0) * 1000
    return result


In [20]:


# ===========================================================================
# SECTION 7 -- COMPARATIVE RUNNER
# ===========================================================================

def run_all_configs(
    prompt: str,
    pipe:         StableDiffusionPipeline,
    pipe_img2img: StableDiffusionImg2ImgPipeline,
    clip:         CLIPEmbedder,
    db:           RetrievalDatabase,
    config:       RAGDiffusionConfig,
    seed:         int = 42,
) -> Dict[str, GenerationResult]:
    """
    Run all five RAG-Diffusion configurations for one prompt and compare.

    Saves generated images to config.OUTPUT_DIR for visual comparison.
    """
    print("\n" + "#" * 78)
    print(f"PROMPT: {prompt}")
    print(f"DB size: {db.n_images()} images  | k={config.N_NEIGHBORS}")
    print("#" * 78)

    runners = {
        "Config1_Baseline_TextOnly":               lambda p: run_config1_baseline_text_only(
            p, pipe, config, seed),
        "Config2_CLIP_kNN_Retrieval":              lambda p: run_config2_clip_knn_conditioning(
            p, pipe, clip, db, config, seed),
        "Config3_Retrieve_Fuse":                   lambda p: run_config3_retrieve_fuse(
            p, pipe_img2img, clip, db, config, seed),
        "Config4_ImageRAG_Adaptive":               lambda p: run_config4_imagrag_adaptive(
            p, pipe, pipe_img2img, clip, db, config, seed),
        "Config5_Full_MultiModal_RetrievalGuided": lambda p: run_config5_full_retrieval_guided(
            p, pipe, pipe_img2img, clip, db, config, seed),
    }

    results: Dict[str, GenerationResult] = {}
    for label, fn in runners.items():
        print(f"\n{'='*62}\nRUNNING: {label}\n{'='*62}")
        try:
            res = fn(prompt)
            res.print_summary()
            res.save(config.OUTPUT_DIR, suffix=f"_{seed}")
            results[label] = res
        except Exception as exc:
            logger.error("Config %s failed: %s", label, exc, exc_info=True)

    print("\n" + "=" * 78)
    print("RAG-DIFFUSION COMPARISON SUMMARY")
    print(f"{'Config':<47} {'Neighbors':>9} {'Ret_ms':>7} {'Gen_ms':>7} {'Total_ms':>9}")
    print("-" * 78)
    for lbl, r in results.items():
        print(f"{lbl:<47} {len(r.neighbors):>9} "
              f"{r.retrieval_ms:>7.0f} {r.generation_ms:>7.0f} {r.total_ms:>9.0f}")
    print("=" * 78 + "\n")
    return results



In [21]:
 """
    End-to-end RAG-Diffusion pipeline orchestrator.

    PHASE 1: Infrastructure loading (run once, heavy).
        - Download demo images for visual database (~8 images).
        - Build CLIP embeddings + FAISS index (the retrieval database).
        - Load Stable Diffusion v2.1-base + img2img pipeline (~5 GB VRAM).
        - Load CLIP ViT-L/14 (~1.7 GB RAM).

    PHASE 2: Per-prompt generation (all five configs).
        Config 1: ~10s, 0 retrieval calls.
        Config 2: ~10s, 1 FAISS query (~20ms).
        Config 3: ~12s, 1 FAISS query + image load (~50ms).
        Config 4: ~20s, 1 FAISS query + 2 generation passes.
        Config 5: ~22s, 2 FAISS queries + adaptive generation.

    PRODUCTION NOTES:
        GPU: A100 (40 GB) handles all configs at float16 in <15s.
        CPU: Usable but slow (3-5 min per config per image). Use --device cpu.
        Database scale: swap DEMO_IMAGE_URLS for a domain-specific corpus.
            Natural images:  LAION-5B (~1B images), OpenImages (~9M images).
            Art styles:      WikiArt (~80K paintings), ArtBench (~60K paintings).
            Domain-specific: medical, satellite, microscopy corpora.
        Swapping the database DOES NOT require retraining the model:
            This is the core value proposition of RDM's semi-parametric design.
    """

"\n   End-to-end RAG-Diffusion pipeline orchestrator.\n\n   PHASE 1: Infrastructure loading (run once, heavy).\n       - Download demo images for visual database (~8 images).\n       - Build CLIP embeddings + FAISS index (the retrieval database).\n       - Load Stable Diffusion v2.1-base + img2img pipeline (~5 GB VRAM).\n       - Load CLIP ViT-L/14 (~1.7 GB RAM).\n\n   PHASE 2: Per-prompt generation (all five configs).\n       Config 1: ~10s, 0 retrieval calls.\n       Config 2: ~10s, 1 FAISS query (~20ms).\n       Config 3: ~12s, 1 FAISS query + image load (~50ms).\n       Config 4: ~20s, 1 FAISS query + 2 generation passes.\n       Config 5: ~22s, 2 FAISS queries + adaptive generation.\n\n   PRODUCTION NOTES:\n       GPU: A100 (40 GB) handles all configs at float16 in <15s.\n       CPU: Usable but slow (3-5 min per config per image). Use --device cpu.\n       Database scale: swap DEMO_IMAGE_URLS for a domain-specific corpus.\n           Natural images:  LAION-5B (~1B images), OpenIma

In [35]:
config = RAGDiffusionConfig()
Path(config.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


In [23]:

logger.info("=== RAG with Diffusion Models Pipeline Starting ===")
logger.info("Device: %s | Dtype: %s", config.DEVICE, config.DTYPE)


2026-05-26 07:37:35 | INFO     | rag_diffusion | === RAG with Diffusion Models Pipeline Starting ===
2026-05-26 07:37:35 | INFO     | rag_diffusion | Device: cuda | Dtype: torch.float16


In [26]:

# --- CLIP embedder (shared across all configs) -----------------------
clip = CLIPEmbedder(config.CLIP_MODEL_ID, config.DEVICE, config.DTYPE)


2026-05-26 07:42:41 | INFO     | rag_diffusion | Loading CLIP: openai/clip-vit-large-patch14 on cuda ...
2026-05-26 07:42:41 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-26 07:42:41 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-large-patch14/32bd64288804d66eefd0ccbe215aa642df71cc41/config.json "HTTP/1.1 200 OK"
2026-05-26 07:42:42 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights: 100%|██████████| 590/590 [00:00<00:00, 4067.48it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-26 07:42:52 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-large-patch14/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-26 07:42:52 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-05-26 07:42:53 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
2026-05-26 07:42:53 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-05-26 07:42:53 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/audio_tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-05-26 07:42:54 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/openai/clip

In [31]:

# --- Build visual retrieval database ---------------------------------
logger.info("Building retrieval database ...")
image_data = download_demo_images(config)
if not image_data:
    raise RuntimeError("No demo images available. Check network access.")
db = build_retrieval_database(image_data, clip, config)
logger.info("Database ready: %d images", db.n_images())


2026-05-26 07:44:54 | INFO     | rag_diffusion | Building retrieval database ...
2026-05-26 07:44:54 | INFO     | rag_diffusion | Downloading: sunset_mountains.jpg
2026-05-26 07:44:56 | INFO     | rag_diffusion | Downloading: forest_path.jpg
2026-05-26 07:44:58 | INFO     | rag_diffusion | Downloading: ocean_waves.jpg
2026-05-26 07:44:59 | INFO     | rag_diffusion | Downloading: city_night.jpg
2026-05-26 07:45:02 | INFO     | rag_diffusion | Downloading: abstract_art.jpg
2026-05-26 07:45:03 | INFO     | rag_diffusion | Downloading: portrait_painting.jpg
2026-05-26 07:45:05 | INFO     | rag_diffusion | Downloading: landscape_oil.jpg
2026-05-26 07:45:07 | INFO     | rag_diffusion | Downloading: architecture.jpg
2026-05-26 07:45:09 | INFO     | rag_diffusion | Demo images loaded: 8
2026-05-26 07:45:09 | INFO     | rag_diffusion | Building retrieval database from 8 images ...
2026-05-26 07:45:21 | INFO     | rag_diffusion |   Embedded 0 / 8 images ...
2026-05-26 07:45:21 | INFO     | rag_d

In [36]:
# --- Load Stable Diffusion pipelines ---------------------------------
pipe         = load_sd_pipeline(config)
pipe_img2img = load_img2img_pipeline(config)


2026-05-26 07:45:54 | INFO     | rag_diffusion | Loading Stable Diffusion: runwayml/stable-diffusion-v1-5 ...
2026-05-26 07:45:55 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/runwayml/stable-diffusion-v1-5 "HTTP/1.1 307 Temporary Redirect"
2026-05-26 07:45:55 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/stable-diffusion-v1-5/stable-diffusion-v1-5 "HTTP/1.1 200 OK"
2026-05-26 07:45:55 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/model_index.json "HTTP/1.1 307 Temporary Redirect"
2026-05-26 07:45:55 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/model_index.json "HTTP/1.1 307 Temporary Redirect"
2026-05-26 07:45:56 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/stable-diffusion-v1-5/stable-diffusion-v1-5/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/model_index.jso

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

2026-05-26 07:45:57 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/feature_extractor/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-26 07:45:57 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder/model.safetensors "HTTP/1.1 307 Temporary Redirect"
2026-05-26 07:45:58 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/tokenizer/merges.txt "HTTP/1.1 307 Temporary Redirect"
2026-05-26 07:45:58 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-26 07:45:58 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/runwaym

Fetching 14 files:   7%|▋         | 1/14 [00:01<00:14,  1.13s/it]

2026-05-26 07:45:58 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/stable-diffusion-v1-5/stable-diffusion-v1-5/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/scheduler%2Fscheduler_config.json "HTTP/1.1 200 OK"
2026-05-26 07:45:58 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/stable-diffusion-v1-5/stable-diffusion-v1-5/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker%2Fconfig.json "HTTP/1.1 200 OK"
2026-05-26 07:45:58 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/stable-diffusion-v1-5/stable-diffusion-v1-5/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/tokenizer%2Fspecial_tokens_map.json "HTTP/1.1 200 OK"


Fetching 14 files:  21%|██▏       | 3/14 [00:01<00:04,  2.59it/s]

2026-05-26 07:45:58 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/stable-diffusion-v1-5/stable-diffusion-v1-5/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder%2Fconfig.json "HTTP/1.1 200 OK"
2026-05-26 07:45:58 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/stable-diffusion-v1-5/stable-diffusion-v1-5/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/tokenizer%2Ftokenizer_config.json "HTTP/1.1 200 OK"
2026-05-26 07:45:58 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/stable-diffusion-v1-5/stable-diffusion-v1-5/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/tokenizer%2Fmerges.txt "HTTP/1.1 200 OK"
2026-05-26 07:45:58 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/tokenizer/vocab.json "HTTP/1.1 307 Temporary Redirect"
2026-05-26 07:45:58 | INFO     | httpx | HTTP Request: GET https://h

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 1784.45it/s]]
CLIPTextModel LOAD REPORT from: C:\Users\dhanu\.cache\huggingface\hub\models--runwayml--stable-diffusion-v1-5\snapshots\451f4fe16113bff5a5d2269ed5ad43b0592e9a14\text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading pipeline components...: 100%|██████████| 6/6 [00:08<00:00,  1.39s/it]


2026-05-26 07:49:35 | INFO     | rag_diffusion |   xformers memory efficient attention enabled.
2026-05-26 07:49:35 | INFO     | rag_diffusion | Stable Diffusion loaded on cuda (dtype=torch.float16)
2026-05-26 07:49:35 | INFO     | rag_diffusion | Loading SD img2img pipeline ...
2026-05-26 07:49:35 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/runwayml/stable-diffusion-v1-5 "HTTP/1.1 307 Temporary Redirect"
2026-05-26 07:49:36 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/stable-diffusion-v1-5/stable-diffusion-v1-5 "HTTP/1.1 200 OK"
2026-05-26 07:49:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/model_index.json "HTTP/1.1 307 Temporary Redirect"
2026-05-26 07:49:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/model_index.json "HTTP/1.1 307 Temporary Redirect"
2026-05-26 07:49:36 | INFO     | httpx

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 488.79it/s]s]
CLIPTextModel LOAD REPORT from: C:\Users\dhanu\.cache\huggingface\hub\models--runwayml--stable-diffusion-v1-5\snapshots\451f4fe16113bff5a5d2269ed5ad43b0592e9a14\text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading pipeline components...: 100%|██████████| 6/6 [00:06<00:00,  1.16s/it]


In [37]:
# --- Demo prompts (test diverse generation scenarios) ----------------
demo_prompts = [
    # Standard prompt: test retrieval augmentation vs baseline
    "a serene mountain landscape at sunset with dramatic clouds",

    # Fine-grained concept: rare object the model may not know well
    "an oil painting of a coastal village with fishing boats",

    # Domain-specific: tests database swap utility
    "a painting in the style of impressionism showing a garden",

    # Abstract/artistic: tests whether retrieval adds style specificity
    "vivid colors abstract art with geometric shapes and textures",
]

In [38]:

for prompt in demo_prompts[:2]:   # run first 2 for demo (4 prompts * 5 configs is expensive)
    run_all_configs(prompt, pipe, pipe_img2img, clip, db, config, seed=42)

logger.info("=== RAG-Diffusion Pipeline Demo Complete ===")
logger.info("Generated images saved to: %s", config.OUTPUT_DIR)



##############################################################################
PROMPT: a serene mountain landscape at sunset with dramatic clouds
DB size: 8 images  | k=5
##############################################################################

RUNNING: Config1_Baseline_TextOnly
2026-05-26 07:49:58 | INFO     | rag_diffusion | Config1: Generating 'a serene mountain landscape at sunset with dramati ...' (no retrieval)


100%|██████████| 50/50 [09:49<00:00, 11.79s/it]



STRATEGY: Config1_Baseline_TextOnly
PROMPT  : a serene mountain landscape at sunset with dramatic clouds
Latency : ret=0ms  gen=606268ms  total=606273ms

2026-05-26 08:00:05 | INFO     | rag_diffusion | Saved: rag_diffusion_outputs\Config1_Baseline_TextOnly_a_serene_mountain_landscape_at_sunset_wi_42.png

RUNNING: Config2_CLIP_kNN_Retrieval
2026-05-26 08:00:05 | INFO     | rag_diffusion | Config2: Retrieved 5 neighbors for 'a serene mountain landscape at sunset wi ...' (top sim=0.186, ret=620ms)


100%|██████████| 50/50 [11:05<00:00, 13.32s/it]



STRATEGY: Config2_CLIP_kNN_Retrieval
PROMPT  : a serene mountain landscape at sunset with dramatic clouds
Neighbors (5):
  [3] sim=0.186  city_night.jpg
  [0] sim=0.169  sunset_mountains.jpg
  [4] sim=0.155  abstract_art.jpg
Latency : ret=620ms  gen=684799ms  total=685426ms

2026-05-26 08:11:30 | INFO     | rag_diffusion | Saved: rag_diffusion_outputs\Config2_CLIP_kNN_Retrieval_a_serene_mountain_landscape_at_sunset_wi_42.png

RUNNING: Config3_Retrieve_Fuse
2026-05-26 08:11:31 | INFO     | rag_diffusion | Config3: Retrieved init image 'city_night.jpg' (sim=0.186, ret=301ms)


100%|██████████| 40/40 [09:45<00:00, 14.65s/it]



STRATEGY: Config3_Retrieve_Fuse_img2img
PROMPT  : a serene mountain landscape at sunset with dramatic clouds
Neighbors (1):
  [3] sim=0.186  city_night.jpg
Latency : ret=301ms  gen=618451ms  total=618760ms

2026-05-26 08:21:49 | INFO     | rag_diffusion | Saved: rag_diffusion_outputs\Config3_Retrieve_Fuse_img2img_a_serene_mountain_landscape_at_sunset_wi_42.png

RUNNING: Config4_ImageRAG_Adaptive
2026-05-26 08:21:50 | INFO     | rag_diffusion | Config4: Diverse reranked 8 candidates -> top 5 (sims=[0.186, 0.155, 0.169], ret=869ms)


  8%|▊         | 2/25 [00:29<05:34, 14.52s/it]

2026-05-26 08:22:20 | ERROR    | rag_diffusion | Config Config4_ImageRAG_Adaptive failed: CUDA error: CUBLAS_STATUS_EXECUTION_FAILED when calling `cublasGemmEx( handle, opa, opb, m, n, k, alpha_ptr, a, CUDA_R_16F, lda, b, CUDA_R_16F, ldb, beta_ptr, c, std::is_same_v<C_Dtype, float> ? CUDA_R_32F : CUDA_R_16F, ldc, compute_type, CUBLAS_GEMM_DEFAULT_TENSOR_OP)`
Traceback (most recent call last):
  File "C:\Users\dhanu\AppData\Local\Temp\ipykernel_12972\3059373969.py", line 41, in run_all_configs
    res = fn(prompt)
          ^^^^^^^^^^
  File "C:\Users\dhanu\AppData\Local\Temp\ipykernel_12972\3059373969.py", line 31, in <lambda>
    "Config4_ImageRAG_Adaptive":               lambda p: run_config4_imagrag_adaptive(
                                                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dhanu\AppData\Local\Temp\ipykernel_12972\645028060.py", line 105, in run_config4_imagrag_adaptive
    draft_out = pipe(
                ^^^^^
  File "c:\Users\dhanu\miniconda


RUNNING: Config5_Full_MultiModal_RetrievalGuided
2026-05-26 08:22:20 | ERROR    | rag_diffusion | Config Config5_Full_MultiModal_RetrievalGuided failed: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
Traceback (most recent call last):
  File "C:\Users\dhanu\AppData\Local\Temp\ipykernel_12972\3059373969.py", line 41, in run_all_configs
    res = fn(prompt)
          ^^^^^^^^^^
  File "C:\Users\dhanu\AppData\Local\Temp\ipykernel_12972\3059373969.py", line 33, in <lambda>
    "Config5_Full_MultiModal_RetrievalGuided": lambda p: run_config5_full_retrieval_guided(
                                                         ^^^^^^^^^^^^^^^^^^^